In [32]:
import os
import time
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight

In [33]:
# 1. Get the notebook directory
notebook_dir = os.getcwd()

# 2. Go up twice (../../), then into data/processed
# os.path.abspath cleans up the path so it looks pretty in error messages
data_path = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "processed", "pr_snapshots_clean_v2.csv"))

# 3. Read the file
df = pd.read_csv(data_path)
df.isnull().sum()

pr_id                             0
owner                             0
name                              0
number                            0
checkpoint_day                    0
days_elapsed                      0
additions                         0
deletions                         0
changed_files                     0
title_length                      0
author_login                   1438
author_type                    1438
num_labels                        0
created_dow                       0
created_hour                      0
comments_so_far                   0
reviews_so_far                    0
reviewer_assigned                 0
first_response_hours         201360
timeline_data_available           0
merged_before_next                0
additions_norm                    0
deletions_norm                    0
changed_files_norm                0
has_response_yet                  0
repo_key                          0
author_prior_pr_count             0
author_prior_merge_rate     

In [34]:
TARGET_ROWS = 30000
TOLERANCE = 0.15  # skip any single repo that alone would push total too far past target



repo_sizes = df.groupby("repo_key").size().sample(frac=1, random_state=42)
keep_repos, total = [], 0
for repo, size in repo_sizes.items():
    if total >= TARGET_ROWS:
        break
    if total + size > TARGET_ROWS * (1 + TOLERANCE):
        continue
    keep_repos.append(repo)
    total += size

df = df[df.repo_key.isin(keep_repos)].copy()
print(f"{len(keep_repos)} repos, {len(df)} rows")

y = df.pop("merged_before_next")
X = df.select_dtypes(include="number").drop(columns=["pr_id", "number", "timeline_data_available"], errors="ignore")
print(f"shape: {X.shape}, target distribution: {y.value_counts().to_dict()}")

# group split: whole repos held out, never split across train/test
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=df["repo_key"]))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
print(f"train repos: {df.repo_key.iloc[train_idx].nunique()}, test repos: {df.repo_key.iloc[test_idx].nunique()}")

5 repos, 30867 rows
shape: (30867, 24), target distribution: {0: 19349, 1: 11518}
train repos: 4, test repos: 1


In [35]:

# (Keep all your other scoring metric imports here)

# 1. Compute your training sample weights
sw_train = compute_sample_weight(class_weight='balanced', y=y_train)

# 2. FIX: Impute missing values for BOTH train and test splits
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

models = {
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Extra Trees": ExtraTreesClassifier(random_state=42),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1)
}

results = []

for name, model in models.items():
    start_time = time.time()
    
    # 3. FIX: Fit and predict using the IMPUTED dataframes
    model.fit(X_train_imp, y_train, sample_weight=sw_train)
    
    y_pred = model.predict(X_test_imp)
    y_prob = model.predict_proba(X_test_imp)
    
    roc_auc = roc_auc_score(y_test, y_prob[:, 1]) if len(set(y)) == 2 else roc_auc_score(y_test, y_prob, multi_class='ovr')
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "ROC AUC": roc_auc,
        "F1 Score": f1_score(y_test, y_pred, average='weighted'),
        "Precision": precision_score(y_test, y_pred, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, y_pred, average='weighted', zero_division=0),
        "Time Taken (s)": time.time() - start_time
    })

benchmark_df = pd.DataFrame(results).sort_values("Balanced Accuracy", ascending=False).set_index("Model")
display(benchmark_df.round(4))

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken (s)
Model,,,,,,,
AdaBoost,0.6203,0.6602,0.6811,0.6257,0.7025,0.6203,1.7559
LightGBM,0.6256,0.6570,0.6929,0.6325,0.6943,0.6256,4.2009
Random Forest,0.6252,0.6392,0.6843,0.6336,0.6714,0.6252,6.4296
XGBoost,0.6114,0.6378,0.6876,0.6191,0.6744,0.6114,0.2532
Extra Trees,0.6276,0.5953,0.6719,0.6289,0.6303,0.6276,4.3861


In [37]:

from sklearn.model_selection import RandomizedSearchCV
# 1. Impute Data
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

# 2. Compute Sample Weights (For everyone except Extra Trees)
sw_train = compute_sample_weight(class_weight='balanced', y=y_train)

# 3. Model & Parameter Configurations
models_config = {
    "AdaBoost": (
        AdaBoostClassifier(random_state=42), 
        {"sample_weight": sw_train}, 
        {"n_estimators": [50, 100, 200], "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0]}
    ),
    "LightGBM": (
        LGBMClassifier(random_state=42, verbose=-1), 
        {"sample_weight": sw_train}, 
        {"max_depth": [3, 5, 7, -1], "learning_rate": [0.01, 0.05, 0.1], "n_estimators": [100, 200, 300], "subsample": [0.7, 0.8, 1.0]}
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=42), 
        {"sample_weight": sw_train}, 
        {"n_estimators": [100, 200, 300], "max_depth": [5, 10, 15, None], "min_samples_split": [2, 5, 10]}
    ),
    "XGBoost": (
        XGBClassifier(eval_metric='logloss', random_state=42), 
        {"sample_weight": sw_train}, 
        {"max_depth": [3, 5, 7], "learning_rate": [0.01, 0.05, 0.1], "n_estimators": [100, 200, 300], "subsample": [0.7, 0.8, 1.0]}
    ),
    "Extra Trees": (
        ExtraTreesClassifier(random_state=42), 
        {},  # Vanilla run: No weights passed
        {"n_estimators": [100, 200, 300], "max_depth": [5, 10, 15, None], "min_samples_split": [2, 5, 10]}
    )
}

# 4. Tuning & Evaluation Loop
results = []

for name, (model, fit_params, params) in models_config.items():
    start_time = time.time()
    
    # Run RandomCV
    search = RandomizedSearchCV(
        estimator=model, 
        param_distributions=params, 
        n_iter=10, 
        scoring="balanced_accuracy", 
        cv=3, 
        random_state=42, 
        n_jobs=-1
    )
    search.fit(X_train_imp, y_train, **fit_params)
    
    # Grab the best tuned model
    best_model = search.best_estimator_
    
    # Generate Predictions
    y_pred = best_model.predict(X_test_imp)
    y_prob = best_model.predict_proba(X_test_imp)
    roc_auc = roc_auc_score(y_test, y_prob[:, 1]) if len(set(y_test)) == 2 else roc_auc_score(y_test, y_prob, multi_class='ovr')
    
    # Store Metrics
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "ROC AUC": roc_auc,
        "F1 Score": f1_score(y_test, y_pred, average='weighted'),
        "Precision": precision_score(y_test, y_pred, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, y_pred, average='weighted', zero_division=0),
        "Time Taken (s)": time.time() - start_time,
        "Best Params": str(search.best_params_)
    })

# 5. Display Final Benchmark Table
benchmark_df = pd.DataFrame(results).sort_values("Balanced Accuracy", ascending=False).set_index("Model")
display(benchmark_df.round(4))

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken (s),Best Params
Model,,,,,,,,
AdaBoost,0.6285,0.6499,0.6690,0.6365,0.6833,0.6285,28.9672,"{'n_estimators': 50, 'learning_rate': 0.5}"
LightGBM,0.6281,0.6494,0.6755,0.6360,0.6829,0.6281,9.6105,"{'subsample': 0.8, 'n_estimators': 200, 'max_d..."
XGBoost,0.6221,0.6455,0.6846,0.6300,0.6802,0.6221,6.7137,"{'subsample': 0.8, 'n_estimators': 100, 'max_d..."
Random Forest,0.6034,0.6327,0.6641,0.6108,0.6710,0.6034,37.3654,"{'n_estimators': 300, 'min_samples_split': 10,..."
Extra Trees,0.6408,0.6100,0.6881,0.6421,0.6436,0.6408,29.2742,"{'n_estimators': 300, 'min_samples_split': 10,..."
